# 03 — Evaluation: FinCrime Investigator

**All outputs are read from local measured experiments; this notebook does not fabricate benchmark results or silently run paid API calls.**

Run from repository root (`jupyter lab` started in `fincrime-investigator`).

## Experimental design

- Experiment A: 12 manually verified FATF retrieval prompts; 9 dev / 3 holdout; BM25, dense, hybrid RRF, hybrid + cross-encoder; Recall@5, MRR@5, Hit@5, warmed latency.
- Experiment B: the original 24 frozen questions; 18 dev / 6 holdout; task-eligible deterministic baseline, FATF-only assistant, financial-tools agent, and agentic RAG; human correctness and evidence review.
- Report audit: independent transaction counts and verified ID existence; **not** independent agent numerical reasoning or laundering-detection F1.
- Labels are never provided to the agent. The network-enriched demonstration subset is not representative of population AML prevalence.


In [ ]:
from pathlib import Path
import json
import pandas as pd
root = Path('data/evaluation')
questions = [json.loads(s) for s in (root/'questions.jsonl').read_text(encoding='utf-8').splitlines() if s.strip()]
print('Frozen investigation questions:',len(questions))
print('By split:', pd.Series([q['split'] for q in questions]).value_counts().to_dict())
print('By category:', pd.Series([q['category'] for q in questions]).value_counts().to_dict())
print('Original gold answers pending:',sum(q.get('gold_answer') is None for q in questions))


## Experiment A — retrieval metrics

Before interpreting scores, check `complete_annotation` and the number of `verified_questions_scored`. Missing labels are *not* counted as unsuccessful retrievals. Keep the test split untouched until development settings are frozen.


In [ ]:
for split in ('dev','test'):
    p = root/f'benchmark_retrieval_metrics_{split}.json'
    if not p.exists():
        print(f'No measured retrieval results yet for {split}: {p}')
        continue
    summary = json.loads(p.read_text(encoding='utf-8'))
    print(f'{split}: full annotation = {summary["complete_annotation"]}; unverified = {summary["unverified_question_ids"]}')
    df = pd.DataFrame.from_dict(summary['metrics'], orient='index')
    display(df)


## Experiment B — report consistency audit

Both agent report schemas copy numerical fields from the deterministic baseline. Numerical agreement checks the system's evidence boundary, **not** model improvement. Exact FATF chunk IDs identify a source, but do not establish that the passage supports the generated explanation.


In [ ]:
for split in ('dev','test'):
    p = root/'benchmark_investigation'/f'report_audit_summary_{split}.json'
    if p.exists():
        audit=json.loads(p.read_text(encoding='utf-8'))
        print('Report audit:',split)
        display(pd.DataFrame.from_dict(audit['by_arm'], orient='index'))
    else:
        print('No report audit results yet for',split)


## Experiment B — answer-level human benchmark

The original 24 questions are only fully evaluated after human verification of gold answers, financial evidence and FATF citations. Use `src.benchmark_qa` to run eligible arms and fill `human_scores_<split>.csv`. A blank score is **missing**, not an error or a zero.


In [ ]:
for split in ('dev','test'):
    p = root/'benchmark_qa'/f'human_summary_{split}.json'
    if p.exists():
        result = json.loads(p.read_text(encoding='utf-8'))
        print('Human-graded question results:', split)
        flattened = {arm: {key: (val.get('mean') if isinstance(val,dict) else val) for key,val in metrics.items()} for arm,metrics in result['per_arm'].items()}
        display(pd.DataFrame.from_dict(flattened, orient='index'))
        print('Denominators:', {arm: {k:v['n'] for k,v in row.items() if isinstance(v,dict)} for arm,row in result['per_arm'].items()})
    else:
        print('No graded question-level results yet for',split)


## Failure analysis and methodological limitations

Review each incorrect or unsupported response, mark failure type (retrieval miss, wrong tool, faulty arithmetic, unsupported claim, citation mismatch, failure to abstain, timeout, insufficient context). Distinguish structural citation validity from claim-level faithfulness. Keep a low-signal case to show proper restraint, and choose one higher-activity case for the final live demo.


In [ ]:
for split in ('dev','test'):
    p = root/'benchmark_qa'/f'human_scores_{split}.csv'
    if not p.exists():
        continue
    reviews = pd.read_csv(p)
    annotated = reviews[reviews['failure_type'].notna() & (reviews['failure_type'].astype(str).str.strip()!='')]
    print(f'{split}: documented failures: {len(annotated)}')
    if len(annotated):
        display(annotated[['question_id','arm','failure_type','review_notes']])


## Conclusions (fill only after measurements)

1. **Retrieval**: Which method led on your verified holdout questions? What was the extra time cost of reranking? Did it actually improve Recall@5/MRR@5?
2. **Investigations**: Did the LLM add faithful qualitative analysis beyond the deterministic report? Was its extra cost and latency justified?
3. **Limitations**: The corpus is three selected FATF PDFs; the transaction dataset is synthetic and demo cases are deliberately label-enriched; small holdout sizes and overlap may limit generalizability. No calibrated criminal-risk probabilities are reported.
4. **Reproducibility**: Record dataset SHA-256, number of chunks, model versions, query splits and hyperparameters in the final README.
